In [ ]:
!pip install roboflow ultralytics opencv-python numpy

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 85.3/85.3 kB 3.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 66.8/66.8 kB 4.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 49.9/49.9 MB 13.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 32.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 7.8/7.8 MB 63.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 363.4/363.4 MB 4.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 13.8/13.8 MB 62.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 24.6/24.6 MB 56.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 883.7/883.7 kB 34.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 664.8/664.8 MB 1.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 211.5/211.5 MB 6.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 56.3/56.3 MB 15.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 127.9/127

In [ ]:
# Install necessary libraries
!pip install -q roboflow ultralytics opencv-python-headless

import cv2
from roboflow import Roboflow
from google.colab import files
from google.colab import drive  # To mount Google Drive
from IPython.display import HTML, display
import numpy as np
import shutil  # To move the file to Google Drive

# ====== MODEL INITIALIZATION ======
print("🔍 Initializing Roboflow model...")

try:
    # Initialize with your API key
    rf = Roboflow(api_key="hHVeZIfvznjKbCGiOglb")
    # Load project - verify exact name from Roboflow Universe
    project = rf.workspace("aegis").project("pothole-detection-i00zy")
    # Try different versions if needed
    for version_num in [10, 1, 2, 3]:  # Test multiple possible versions
        try:
            model = project.version(version_num).model
            # Test prediction with a small black image
            test_pred = model.predict(np.zeros((100, 100, 3), dtype=np.uint8), confidence=40).json()
            if 'predictions' in test_pred:
                print(f"✅ Successfully loaded version {version_num}")
                break
        except:
            continue
    else:
        raise RuntimeError("Could not find a working model version")

except Exception as e:
    print(f"❌ Failed to initialize model: {e}")
    print("\n🛠️ Trying alternative approach: Downloading model weights...")
    # Fallback: Download model weights
    !pip install -q torch torchvision
    project.version(1).deploy("yolov8")
    from ultralytics import YOLO
    model = YOLO("yolov8n.pt")
    print("✅ Loaded local YOLOv8 model")

# ====== VIDEO PROCESSING ======
print("\n📤 Please upload your video file:")
uploaded = files.upload()
input_path = list(uploaded.keys())[0]
output_path = "processed_" + input_path

# Video setup
cap = cv2.VideoCapture(input_path)
if not cap.isOpened():
    raise ValueError("❌ Could not open video file")

fps = int(cap.get(cv2.CAP_PROP_FPS))
width = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
height = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))

# Output video
fourcc = cv2.VideoWriter_fourcc(*'mp4v')
out = cv2.VideoWriter(output_path, fourcc, fps, (width, height))

frame_count = 0
total_potholes = 0
print(f"\n🎞️ Processing video ({width}x{height}, {fps} FPS)...")

# Processing loop
while cap.isOpened():
    ret, frame = cap.read()
    if not ret:
        break
    frame_count += 1
    # Only process every 3rd frame for speed
    if frame_count % 3 == 0:
        try:
            # Convert and predict
            rgb_frame = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
            if 'predict' in dir(model):  # Roboflow model
                predictions = model.predict(rgb_frame, confidence=40).json()
                detections = predictions.get('predictions', [])
            else:  # YOLO model
                results = model(rgb_frame)
                detections = results[0].boxes.data.cpu().numpy()

            # Process detections
            for det in detections:
                if 'predict' in dir(model):  # Roboflow format
                    x = int(det['x'] - det['width'] / 2)
                    y = int(det['y'] - det['height'] / 2)
                    w = int(det['width'])
                    h = int(det['height'])
                    conf = det['confidence']
                else:  # YOLO format
                    x, y, w, h, conf = map(int, det[:5])

                cv2.rectangle(frame, (x, y), (x + w, y + h), (0, 255, 0), 2)
                cv2.putText(frame, f"Pothole: {conf:.2f}",
                            (x, y - 10), cv2.FONT_HERSHEY_SIMPLEX, 0.6, (0, 255, 0), 2)
                total_potholes += 1
        except Exception as e:
            print(f"⚠️ Frame {frame_count} error: {str(e)[:50]}...")

    out.write(frame)

# Cleanup
cap.release()
out.release()

# Results
print(f"\n✅ Processing complete! Saved to '{output_path}'")

# Mount Google Drive (for Colab users)
drive.mount('/content/drive')

# Move processed video to Google Drive (Optional, but recommended)
shutil.move(output_path, '/content/drive/My Drive/processed_video.mp4')
print(f"✅ Video saved to Google Drive: /content/drive/My Drive/processed_video.mp4")


print("\n📊 Detection Summary:")
print(f"Total frames processed: {frame_count}")
print(f"Total potholes detected: {total_potholes}")


🔍 Initializing Roboflow model...
loading Roboflow workspace...
loading Roboflow project...
✅ Successfully loaded version 2

📤 Please upload your video file:


Saving mixkit-potholes-in-a-rural-road-25208-hd-ready.mp4 to mixkit-potholes-in-a-rural-road-25208-hd-ready (6).mp4

🎞️ Processing video (1280x720, 25 FPS)...

✅ Processing complete! Saved to 'processed_mixkit-potholes-in-a-rural-road-25208-hd-ready (6).mp4'
Mounted at /content/drive
✅ Video saved to Google Drive: /content/drive/My Drive/processed_video.mp4



📊 Detection Summary:
Total frames processed: 692
Total potholes detected: 1497
